# Silver Phase: Complete Transformation Pipeline
Unified notebook combining: Deduplication → Standardization → PII Masking → Feature Engineering

## Initialize & Load Bronze Data

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from datetime import datetime
import pandas as pd

spark = SparkSession.builder.appName("silver-complete").getOrCreate()

print("\n=== SILVER LAYER - COMPLETE TRANSFORMATION ===")
print(f"Start time: {datetime.now()}")

# Load bronze data
sdf = spark.read.table("data_engineering_workshop.creditcard.creditcard_bronze")
initial_count = sdf.count()
print(f"\nLoaded bronze layer: {initial_count:,} rows")

## Step 1: Deduplication

In [ ]:
print(f"\n=== STEP 1: DEDUPLICATION ===")

# Detect exact duplicates
window = Window.partitionBy(
    "time", "v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8", "v9", "v10",
    "v11", "v12", "v13", "v14", "v15", "v16", "v17", "v18", "v19", "v20",
    "v21", "v22", "v23", "v24", "v25", "v26", "v27", "v28", "amount", "class"
).orderBy("load_timestamp")

sdf = sdf.withColumn("rn", F.row_number().over(window))
exact_dups = sdf.filter(F.col("rn") > 1).count()
sdf = sdf.filter(F.col("rn") == 1).drop("rn")

print(f"Exact duplicates removed: {exact_dups:,}")

# Detect proxy duplicates (same amount + class within 5 min)
time_window_sec = 300
window = Window.orderBy("time").partitionBy("amount", "class").rangeBetween(-time_window_sec, 0)
sdf = sdf.withColumn("dup_count", F.count("time").over(window))
proxy_dups = sdf.filter(F.col("dup_count") > 1).count()
sdf = sdf.withColumn(
    "is_proxy_duplicate",
    F.when(F.col("dup_count") > 1, True).otherwise(False)
)
sdf = sdf.drop("dup_count")

print(f"Proxy duplicates flagged: {proxy_dups:,}")

# Write dedup table
sdf.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_silver_dedup"
)
print(f"✓ Dedup table written: {sdf.count():,} rows")

## Step 2: Standardization

In [ ]:
print(f"\n=== STEP 2: STANDARDIZATION ===")

# Load dedup data
sdf = spark.read.table("data_engineering_workshop.creditcard.creditcard_silver_dedup")

# Create time features
sdf = sdf.withColumn(
    "seconds_mod",
    F.col("time") % (24 * 3600)
)

sdf = sdf.withColumn(
    "hour_of_day",
    F.floor(F.col("seconds_mod") / 3600).cast("int")
)

sdf = sdf.withColumn(
    "day_num",
    F.floor(F.col("time") / (24 * 3600)).cast("int")
)

sdf = sdf.withColumn(
    "is_night_hour",
    (F.col("hour_of_day") >= 22) | (F.col("hour_of_day") < 6)
)

sdf = sdf.drop("seconds_mod")

# Standardize amount
amount_stats = sdf.agg(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.avg("amount").alias("avg_amount"),
    F.stddev("amount").alias("std_amount")
).collect()[0]

min_amt = amount_stats['min_amount']
max_amt = amount_stats['max_amount']
avg_amt = amount_stats['avg_amount']
std_amt = amount_stats['std_amount']
range_amt = max_amt - min_amt

print(f"Amount range: ${min_amt:.2f} - ${max_amt:.2f}")

sdf = sdf.withColumn(
    "amount_normalized",
    (F.col("amount") - min_amt) / range_amt
)

# Flag high amounts (top 5%)
amount_p95 = sdf.approxQuantile("amount", [0.95], 0.01)[0]
sdf = sdf.withColumn(
    "is_high_amount",
    F.col("amount") > amount_p95
)

# Write standardized table
sdf.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_silver_standardized"
)
print(f"✓ Standardized table written: {sdf.count():,} rows")

## Step 3: PII Masking & Data Governance

In [ ]:
print(f"\n=== STEP 3: PII MASKING ===")

# Load standardized data
sdf = spark.read.table("data_engineering_workshop.creditcard.creditcard_silver_standardized")

# Amount buckets
sdf = sdf.withColumn(
    "amount_bucket",
    F.when(F.col("amount") == 0, "zero")
      .when((F.col("amount") > 0) & (F.col("amount") <= 50), "micro")
      .when((F.col("amount") > 50) & (F.col("amount") <= 200), "small")
      .when((F.col("amount") > 200) & (F.col("amount") <= 1000), "medium")
      .when(F.col("amount") > 1000, "large")
      .otherwise("unknown")
)

# Data classification tags
sdf = sdf.withColumn("amount_classification", F.lit("SENSITIVE"))
sdf = sdf.withColumn("features_classification", F.lit("PUBLIC"))
sdf = sdf.withColumn("class_classification", F.lit("SENSITIVE"))

# Audit columns
current_ts = datetime.now()
sdf = sdf.withColumn(
    "pii_masked_timestamp",
    F.lit(current_ts).cast("timestamp")
)
sdf = sdf.withColumn("data_version", F.lit("1.0"))
sdf = sdf.withColumn("processing_stage", F.lit("silver_masked"))

# Flag high-value transactions requiring monitoring
sdf = sdf.withColumn(
    "requires_monitoring",
    F.col("is_high_amount") | F.col("is_night_hour")
)

# Write masked table
sdf.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_silver_masked"
)
print(f"✓ Masked table written: {sdf.count():,} rows")

## Step 4: Feature Engineering

In [ ]:
print(f"\n=== STEP 4: FEATURE ENGINEERING ===")

# Load masked data
sdf = spark.read.table("data_engineering_workshop.creditcard.creditcard_silver_masked")

# Risk indicators
amount_p95 = sdf.approxQuantile("amount", [0.95], 0.01)[0]

sdf = sdf.withColumn(
    "high_amount_risk",
    F.col("amount") > amount_p95
)

sdf = sdf.withColumn(
    "unusual_time_risk",
    F.col("is_night_hour")
)

risk_score = (
    F.when(F.col("high_amount_risk"), 1).otherwise(0) +
    F.when(F.col("unusual_time_risk"), 1).otherwise(0) +
    F.when(F.col("is_high_amount"), 1).otherwise(0) +
    F.when(F.col("is_proxy_duplicate"), 1).otherwise(0)
)

sdf = sdf.withColumn("risk_score", risk_score)

sdf = sdf.withColumn(
    "risk_level",
    F.case()
      .when(F.col("risk_score") >= 3, "CRITICAL")
      .when(F.col("risk_score") == 2, "HIGH")
      .when(F.col("risk_score") == 1, "MEDIUM")
      .otherwise("LOW")
)

# Feature interactions
sdf = sdf.withColumn("v18_v19_interaction", F.col("v18") * F.col("v19"))
sdf = sdf.withColumn("v18_v20_interaction", F.col("v18") * F.col("v20"))
sdf = sdf.withColumn("v18_amount_interaction", F.col("v18") * F.col("amount_normalized"))

sdf = sdf.withColumn("high_v18_flag", F.abs(F.col("v18")) > 2.0)
sdf = sdf.withColumn("high_v19_flag", F.abs(F.col("v19")) > 2.0)

# Time series features
window_order = Window.orderBy("time")

sdf = sdf.withColumn(
    "transaction_order",
    F.row_number().over(window_order)
)

sdf = sdf.withColumn(
    "time_since_prev",
    F.col("time") - F.lag("time", 1, 0).over(window_order)
)

sdf = sdf.withColumn(
    "time_gap_category",
    F.case()
      .when(F.col("time_since_prev") == 0, "immediate")
      .when(F.col("time_since_prev") <= 60, "short")
      .when(F.col("time_since_prev") <= 300, "medium")
      .otherwise("long")
)

# Write final table
sdf.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_silver_final"
)
final_count = sdf.count()
print(f"✓ Final silver table written: {final_count:,} rows")

print(f"\n=== SILVER LAYER COMPLETE ===")
print(f"Rows: {initial_count:,} → {final_count:,}")
print(f"Features engineered: {len(sdf.columns)} columns")